<a href="https://colab.research.google.com/github/yoseph1129/data-structure-lab4/blob/master/py2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# -------------------------------------------------------------------------
# [코랩 환경 버그 방지 코드]
# 코랩 세션 초기화로 인해 case.txt가 삭제되어 FileNotFoundError가 발생하는 것을
# 원천 방지하고자, 실행 시 오리지널 텍스트 파일을 자동으로 생성합니다.
# -------------------------------------------------------------------------
with open('case.txt', 'w', encoding='utf-8') as f:
    f.write("6 7\n")
    f.write("1 0 0 1 0 1 1\n")
    f.write("1 0 0 1 0 0 1\n")
    f.write("1 1 1 1 1 1 1\n")
    f.write("0 1 0 0 1 0 1\n")
    f.write("0 1 0 0 1 0 1\n")
    f.write("1 1 1 1 1 1 1\n")

# -------------------------------------------------------------------------
# 입력 파싱 파트
# input().split()이 코랩에서 꼬여 KeyboardInterrupt나 ValueError를 유발하는
# 고질적 문제를 해결하기 위해, 파일 전체를 읽어(read) 토큰 단위로 분리합니다.
# 이 방식은 텍스트 파일 맨 끝에 공백이나 임의의 숫자가 있어도 안전합니다.
# -------------------------------------------------------------------------
with open('case.txt', 'r', encoding='utf-8') as f:
    input_data = f.read().split()

if input_data:
    N = int(input_data[0])
    M = int(input_data[1])
    print(f"입력된 N(행): {N}, M(열): {M}")

    concerts = []
    idx = 2
    # N행 M열의 크기만큼만 정확히 읽어 도면 배열(concerts)을 생성합니다.
    for _ in range(N):
        row = []
        for _ in range(M):
            row.append(int(input_data[idx]))
            idx += 1
        concerts.append(row)


###################################
def count_stages(concerts):
    """
    [알고리즘 설계 및 구현 설명]

    1. 설계 이유 (Why BFS?):
       - 주어진 격자(Grid) 내에서 펜스(1)로 가로막히지 않고 상하좌우로 연결된
         빈 공간(0)의 덩어리(Connected Component) 개수를 세는 전형적인 그래프 탐색 문제입니다.
       - DFS(깊이 우선 탐색)로도 구현이 가능하지만, 파이썬의 경우 재귀 호출 깊이 제한(RecursionError)
         문제가 발생할 위험이 있습니다. 따라서 데크(deque)를 활용하여 안정적으로 인접 영역을
         모두 탐색할 수 있는 BFS(너비 우선 탐색)를 선택했습니다.

    2. 설계 방법 (How?):
       - 2중 for문으로 N x M 격자를 순회하다가 관객/펜스가 없는 빈 공간('0')을 만나면
         무대의 개수를 1 증가시키고, 해당 좌표를 기점으로 BFS 탐색을 시작합니다.
       - BFS 큐에서 좌표를 꺼내 상하좌우 4방향을 검사하며, 인접한 빈 공간(0)을 모두 찾아냅니다.
       - [공간복잡도 최적화]: 별도의 방문(visited) 확인용 2차원 배열을 만들지 않고,
         방문한 빈 공간('0')을 즉시 펜스('1')로 덮어쓰는(In-place marking) 방식을 사용하여
         추가적인 메모리 사용을 최소화했습니다.

    3. 예외 케이스 처리 (Hidden Cases):
       - 전체가 펜스로만 이루어진 경우(무대 0개), 펜스가 하나도 없는 경우(무대 1개) 모두
         본 알고리즘의 완전 순회 및 덮어쓰기 로직을 통해 완벽하게 정상 처리됩니다.
    """
    from collections import deque

    # 엣지 케이스: 배열이 비어있을 경우 0 반환
    if not concerts or not concerts[0]:
        return 0

    rows = len(concerts)
    cols = len(concerts[0])
    stage_count = 0  # 독립 무대 공간의 개수

    # 4방향 탐색을 위한 델타값 (상, 하, 좌, 우)
    dr = [-1, 1, 0, 0]
    dc = [0, 0, -1, 1]

    # 도면 전체를 순회하며 탐색 기점을 찾습니다.
    for r in range(rows):
        for c in range(cols):
            # 빈 공간(0)을 발견하면 독립된 하나의 무대로 카운트합니다.
            if concerts[r][c] == 0:
                stage_count += 1

                # BFS 큐 초기화 및 시작 위치 방문 처리 (0 -> 1)
                queue = deque([(r, c)])
                concerts[r][c] = 1

                # 연결된 모든 빈 공간을 찾아 방문 처리합니다.
                while queue:
                    curr_r, curr_c = queue.popleft()

                    # 4방향(상하좌우) 검사
                    for i in range(4):
                        next_r = curr_r + dr[i]
                        next_c = curr_c + dc[i]

                        # 도면 범위를 벗어나지 않고, 다음 위치가 빈 공간(0)인 경우
                        if 0 <= next_r < rows and 0 <= next_c < cols:
                            if concerts[next_r][next_c] == 0:
                                # 큐에 넣기 전 즉시 1로 변경하여 메모리 초과 방지
                                concerts[next_r][next_c] = 1
                                queue.append((next_r, next_c))

    # 계산된 총 무대 개수 반환
    return stage_count

print(f"계산된 독립 무대의 총 개수: {count_stages(concerts)}")

입력된 N(행): 6, M(열): 7
계산된 독립 무대의 총 개수: 5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')